# AirPulse V4 — chạy CPU trên Colab

Gói có model sẵn. Dữ liệu bán tổng hợp; kết quả là nghiên cứu/replay, chưa xác nhận tiết kiệm thị trường. Đọc THAY_DOI_V4.md. Nếu Colab yêu cầu khởi động lại sau cài thư viện, hãy restart rồi chạy tiếp các cell import.


In [ ]:
from pathlib import Path
from zipfile import ZipFile
from google.colab import files
import sys, subprocess
uploaded = files.upload()  # Chọn AirPulse_V4_Booking_Model_Package.zip
zips=[Path('/content')/n for n in uploaded if n.endswith('.zip')]
assert zips, 'Cần upload gói ZIP V4'
with ZipFile(zips[0]) as z:
    for entry in z.infolist():
        assert (Path('/content')/entry.filename).resolve().is_relative_to(Path('/content').resolve())
    z.extractall('/content')
ROOT=Path('/content/AirPulse_V4_Booking_Model')
assert (ROOT/'airpulse_v4.py').is_file()
sys.path.insert(0,str(ROOT))
print(ROOT)


In [ ]:
%pip -q install -r /content/AirPulse_V4_Booking_Model/requirements-dev.txt


In [ ]:
from pathlib import Path
import sys, subprocess
ROOT=Path('/content/AirPulse_V4_Booking_Model')
sys.path.insert(0,str(ROOT))
from demo import create_example
request, response = create_example()
import pandas as pd
display(pd.DataFrame(response['forecast_slots']))
print(response['status'], response['recommendation']['action'])


## Huấn luyện lại — tùy chọn
Cell sau tái lập từ CSV đi kèm. Không cần chạy để dùng model sẵn.


In [ ]:
subprocess.run([sys.executable,'train.py'],cwd=ROOT,check=True)
subprocess.run([sys.executable,'demo.py'],cwd=ROOT,check=True)


In [ ]:
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],cwd=ROOT,check=True)


In [ ]:
import json
from google.colab import files
result=json.loads((ROOT/'trained/metrics.json').read_text())
print('Policy:',result['policy'])
print('Test:',result['test']['sequential'])
print('Chưa có kiểm chứng thị trường mới; test này đã được dùng trong phát triển.')
files.download(str(ROOT/'trained/airpulse_v4_model.joblib'))
